# Low-Light Image Enhancement Based on the LIME Research Paper

**Digital Image Processing Project 19**  
**Selected Research Paper:** *LIME: Low-Light Image Enhancement via Illumination Map Estimation*  
**Authors:** Xiaojie Guo, Yu Li, Haibin Ling  
**Published in:** IEEE Transactions on Image Processing, 2017  
**DOI:** 10.1109/TIP.2016.2639450  
**Group Members:** Student 1, Student 2, Student 3

This notebook is organized around one research paper, as required by the project rubric. The main implementation is the LIME methodology, followed by baseline comparison and a small improvement over LIME.


## Rubric Mapping

| Rubric Item | Where It Is Covered |
| --- | --- |
| Problem Understanding | Part A |
| Research Paper Selection | Part B |
| Literature Review | Part C |
| Methodology Understanding | Part D |
| Implementation | Parts E-H |
| DIP Techniques | HE, CLAHE, Retinex, illumination estimation, guided filtering, gamma correction |
| Results and Performance Analysis | Parts I-K |
| Innovation / Improvement | Part H |
| IEEE Report and Viva | Part L |


# Part A - Problem Understanding

Low-light images suffer from low brightness, low contrast, loss of visible detail, color distortion, and noise amplification. These problems are important in surveillance systems, night photography, traffic monitoring, mobile photography, and security applications.

The aim of this project is to enhance low-light images using a research-paper-based classical image processing method, then compare the result with standard enhancement techniques.


# Part B - Selected Research Paper

The selected paper is:

**X. Guo, Y. Li, and H. Ling, "LIME: Low-Light Image Enhancement via Illumination Map Estimation," IEEE Transactions on Image Processing, vol. 26, no. 2, pp. 982-993, 2017.**

Paper links:

- DOI page: https://doi.org/10.1109/TIP.2016.2639450
- Public PDF: https://www3.cs.stonybrook.edu/~hling/publication/LIME-tip.pdf
- arXiv summary: https://arxiv.org/abs/1605.05034

**Reason for selection:** LIME is a high-quality IEEE Transactions paper, directly related to low-light enhancement, and its method is based on Digital Image Processing concepts: illumination estimation, filtering/refinement, gamma correction, and image restoration.


# Part C - Literature Review

| Method | Main Idea | Strength | Limitation |
| --- | --- | --- | --- |
| Global Histogram Equalization | Redistributes global intensity histogram | Simple and fast | Can over-enhance and distort colors |
| CLAHE | Applies local histogram equalization with contrast limiting | Improves local contrast | Parameters must be tuned; may amplify noise |
| Gamma Correction | Nonlinear intensity mapping | Fast brightness improvement | Does not estimate illumination |
| Retinex | Separates illumination and reflectance | Good theoretical model | Sensitive to scale/parameters |
| LIME | Estimates and refines illumination map | Preserves structure and enhances dark regions | Noise may still increase in very dark images |
| Improved LIME | Adds denoising, adaptive gamma, and CLAHE refinement | Better practical visibility | Slightly higher runtime |


# Part D - LIME Paper Methodology

LIME assumes that a low-light image can be enhanced by estimating and correcting its illumination.

**Step 1: Initial Illumination Map**

For each pixel `x`, the initial illumination is estimated as:

`T_hat(x) = max(R(x), G(x), B(x))`

**Step 2: Illumination Refinement**

The raw illumination map is refined to preserve important edges while smoothing noise. The original paper formulates this as a structure-aware optimization problem. In this notebook, the same idea is implemented using guided filtering / edge-preserving smoothing, which is practical in OpenCV.

**Step 3: Illumination Correction**

The enhanced image is obtained as:

`R(x) = I(x) / T(x)^gamma`

where `I(x)` is the input image, `T(x)` is the refined illumination map, and `gamma` controls enhancement strength.

**Step 4: Output Reconstruction**

The output is clipped to `[0, 255]` and converted back to a displayable RGB image.


# Part E - Environment Setup

Run the next cells in Google Colab. If no images are uploaded, the notebook generates synthetic low-light/reference samples so the full workflow can still be demonstrated.


In [ ]:
!pip -q install opencv-python-headless scikit-image matplotlib pandas seaborn pillow


In [ ]:
import cv2
import math
import time
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from skimage.metrics import structural_similarity

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

BASE_DIR = Path("/content/lime_low_light_project") if IN_COLAB else Path("lime_low_light_project")
INPUT_DIR = BASE_DIR / "input_low_light"
REF_DIR = BASE_DIR / "reference_normal_light"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURE_DIR = BASE_DIR / "figures"
TABLE_DIR = BASE_DIR / "tables"

for folder in [BASE_DIR, INPUT_DIR, REF_DIR, OUTPUT_DIR, FIGURE_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", IN_COLAB)
print("Project folder:", BASE_DIR)


## Upload Low-Light Images


In [ ]:
def upload_to_folder(target_folder, label):
    target_folder.mkdir(parents=True, exist_ok=True)
    if not IN_COLAB:
        print(f"Upload skipped outside Colab: {label}")
        return []
    print(f"Upload {label}")
    uploaded = files.upload()
    saved = []
    for name, content in uploaded.items():
        path = target_folder / name
        with open(path, "wb") as f:
            f.write(content)
        saved.append(path)
    return saved

uploaded_inputs = upload_to_folder(INPUT_DIR, "low-light input images")
print("Uploaded low-light images:", len(uploaded_inputs))


## Optional Reference Images


In [ ]:
UPLOAD_REFERENCE_IMAGES = False

if UPLOAD_REFERENCE_IMAGES:
    uploaded_refs = upload_to_folder(REF_DIR, "normal-light reference images with matching filenames")
else:
    uploaded_refs = []
    print("Reference upload skipped. Set UPLOAD_REFERENCE_IMAGES = True if paired references are available.")

print("Uploaded reference images:", len(uploaded_refs))


## Image I/O Helpers


In [ ]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def list_images(folder):
    folder = Path(folder)
    return sorted([p for p in folder.glob("*") if p.suffix.lower() in VALID_EXTS])

def read_rgb(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def save_rgb(path, image_rgb):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    image_rgb = np.clip(image_rgb, 0, 255).astype(np.uint8)
    cv2.imwrite(str(path), cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))

def to_gray(image_rgb):
    return cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

def reference_for(path):
    candidate = REF_DIR / Path(path).name
    return candidate if candidate.exists() else None


## Synthetic Fallback Dataset


In [ ]:
def make_reference(seed, h=320, w=480):
    rng = np.random.default_rng(seed)
    image = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(h):
        image[y, :, 0] = np.clip(35 + 110 * y / h, 0, 255)
        image[y, :, 1] = np.clip(45 + 95 * y / h, 0, 255)
        image[y, :, 2] = np.clip(70 + 80 * y / h, 0, 255)
    for _ in range(18):
        x1 = int(rng.integers(10, w - 100))
        y1 = int(rng.integers(20, h - 80))
        x2 = x1 + int(rng.integers(40, 130))
        y2 = y1 + int(rng.integers(25, 85))
        color = tuple(int(v) for v in rng.integers(60, 220, size=3))
        cv2.rectangle(image, (x1, y1), (x2, y2), color, -1)
    for _ in range(6):
        x = int(rng.integers(35, w - 35))
        y = int(rng.integers(35, h - 35))
        r = int(rng.integers(12, 30))
        cv2.circle(image, (x, y), r, (245, 230, 155), -1)
    return cv2.GaussianBlur(image, (3, 3), 0)

def make_low_light(reference, seed):
    rng = np.random.default_rng(seed + 500)
    h, w = reference.shape[:2]
    illum_x = np.linspace(0.18, 0.55, w, dtype=np.float32)[None, :, None]
    illum_y = np.linspace(0.8, 1.05, h, dtype=np.float32)[:, None, None]
    low = reference.astype(np.float32) * illum_x * illum_y
    low = np.power(np.clip(low / 255.0, 0, 1), 1.55) * 255.0
    low = low + rng.normal(0, 8, low.shape)
    return np.clip(low, 0, 255).astype(np.uint8)

if len(list_images(INPUT_DIR)) == 0:
    for i in range(1, 11):
        ref = make_reference(i)
        low = make_low_light(ref, i)
        name = f"sample_{i:02d}.png"
        save_rgb(INPUT_DIR / name, low)
        save_rgb(REF_DIR / name, ref)
    print("Generated 10 synthetic paired samples.")
else:
    print("Using uploaded images.")

print("Input images:", len(list_images(INPUT_DIR)))
print("Reference images:", len(list_images(REF_DIR)))


# Part F - Baseline Methods


In [ ]:
def global_he(image_rgb):
    ycrcb = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2YCrCb)
    y, cr, cb = cv2.split(ycrcb)
    y = cv2.equalizeHist(y)
    return cv2.cvtColor(cv2.merge([y, cr, cb]), cv2.COLOR_YCrCb2RGB)

def clahe_ycrcb(image_rgb, clip_limit=2.0, tile_grid_size=(8, 8)):
    ycrcb = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2YCrCb)
    y, cr, cb = cv2.split(ycrcb)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    y = clahe.apply(y)
    return cv2.cvtColor(cv2.merge([y, cr, cb]), cv2.COLOR_YCrCb2RGB)

def gamma_correction(image_rgb, gamma=0.65):
    table = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
    return cv2.LUT(image_rgb, table)

def gamma_clahe(image_rgb, gamma=0.65):
    return clahe_ycrcb(gamma_correction(image_rgb, gamma=gamma))

def normalize_float_to_uint8(channel):
    min_v, max_v = np.min(channel), np.max(channel)
    if max_v - min_v < 1e-6:
        return np.zeros_like(channel, dtype=np.uint8)
    return np.clip((channel - min_v) * 255.0 / (max_v - min_v), 0, 255).astype(np.uint8)

def single_scale_retinex(image_rgb, sigma=30):
    image = image_rgb.astype(np.float32) + 1.0
    outputs = []
    for c in range(3):
        blur = cv2.GaussianBlur(image[:, :, c], (0, 0), sigma) + 1.0
        retinex = np.log(image[:, :, c]) - np.log(blur)
        outputs.append(normalize_float_to_uint8(retinex))
    return cv2.merge(outputs)

def multi_scale_retinex(image_rgb, sigmas=(15, 80, 250)):
    image = image_rgb.astype(np.float32) + 1.0
    total = np.zeros_like(image)
    for sigma in sigmas:
        blur = cv2.GaussianBlur(image, (0, 0), sigma) + 1.0
        total += np.log(image) - np.log(blur)
    total /= len(sigmas)
    return cv2.merge([normalize_float_to_uint8(total[:, :, c]) for c in range(3)])

print("Baseline methods ready.")


# Part G - LIME Paper Implementation


The following functions implement the main LIME flow:

1. Estimate initial illumination map by channel maximum.
2. Refine illumination with edge-preserving guided filtering.
3. Enhance the image by dividing by the corrected illumination.
4. Save intermediate maps for methodology explanation.


In [ ]:
def box_filter(image, radius):
    ksize = 2 * radius + 1
    return cv2.boxFilter(image, ddepth=-1, ksize=(ksize, ksize), normalize=True)

def guided_filter(guide, src, radius=15, eps=1e-3):
    guide = guide.astype(np.float32)
    src = src.astype(np.float32)
    mean_i = box_filter(guide, radius)
    mean_p = box_filter(src, radius)
    corr_i = box_filter(guide * guide, radius)
    corr_ip = box_filter(guide * src, radius)
    var_i = corr_i - mean_i * mean_i
    cov_ip = corr_ip - mean_i * mean_p
    a = cov_ip / (var_i + eps)
    b = mean_p - a * mean_i
    mean_a = box_filter(a, radius)
    mean_b = box_filter(b, radius)
    q = mean_a * guide + mean_b
    return np.clip(q, 0.05, 1.0)

def lime_initial_illumination(image_rgb):
    image = image_rgb.astype(np.float32) / 255.0
    return np.max(image, axis=2)

def lime_refine_illumination(initial_t, radius=15, eps=1e-3):
    return guided_filter(initial_t, initial_t, radius=radius, eps=eps)

def lime_enhance_with_map(image_rgb, refined_t, gamma=0.8):
    image = image_rgb.astype(np.float32) / 255.0
    corrected_t = np.power(np.clip(refined_t, 0.05, 1.0), gamma)
    enhanced = image / corrected_t[:, :, None]
    enhanced = np.clip(enhanced, 0, 1)
    return (enhanced * 255).astype(np.uint8)

def lime_paper_method(image_rgb, gamma=0.8, radius=15, eps=1e-3, return_debug=False):
    initial_t = lime_initial_illumination(image_rgb)
    refined_t = lime_refine_illumination(initial_t, radius=radius, eps=eps)
    enhanced = lime_enhance_with_map(image_rgb, refined_t, gamma=gamma)
    debug = {
        "initial_illumination": initial_t,
        "refined_illumination": refined_t,
        "gamma": gamma,
        "radius": radius,
        "eps": eps,
    }
    return (enhanced, debug) if return_debug else enhanced

print("LIME paper implementation ready.")


# Part H - Proposed Improvement over LIME


The innovation part is intentionally small and defensible: improve LIME by adding denoising, adaptive gamma, and CLAHE on the luminance channel after LIME enhancement.

This is presented as **Improved LIME**, not as the original paper method.


In [ ]:
def estimate_noise(gray):
    lap = cv2.Laplacian(gray.astype(np.float32), cv2.CV_32F)
    return float(np.median(np.abs(lap - np.median(lap))))

def choose_adaptive_gamma(image_rgb):
    mean_v = np.mean(to_gray(image_rgb))
    if mean_v < 35:
        return 0.55
    if mean_v < 70:
        return 0.70
    return 0.85

def improved_lime_method(image_rgb, return_debug=False):
    denoised = cv2.bilateralFilter(image_rgb, d=7, sigmaColor=55, sigmaSpace=55)
    gamma = choose_adaptive_gamma(denoised)
    lime_output, debug = lime_paper_method(denoised, gamma=gamma, radius=15, eps=1e-3, return_debug=True)
    final_output = clahe_ycrcb(lime_output, clip_limit=1.6, tile_grid_size=(8, 8))
    debug["adaptive_gamma"] = gamma
    debug["noise_score"] = estimate_noise(to_gray(image_rgb))
    debug["denoised_input"] = denoised
    debug["lime_before_clahe"] = lime_output
    return (final_output, debug) if return_debug else final_output

print("Improved LIME method ready.")


# Part I - Evaluation Metrics


In [ ]:
def mse(enhanced, reference):
    return float(np.mean((enhanced.astype(np.float64) - reference.astype(np.float64)) ** 2))

def psnr(enhanced, reference):
    value = mse(enhanced, reference)
    if value <= 1e-12:
        return float("inf")
    return float(20 * math.log10(255.0 / math.sqrt(value)))

def ssim_score(enhanced, reference):
    return float(structural_similarity(to_gray(reference), to_gray(enhanced), data_range=255))

def entropy(image_rgb):
    gray = to_gray(image_rgb)
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).ravel()
    prob = hist / max(1.0, np.sum(hist))
    prob = prob[prob > 0]
    return float(-np.sum(prob * np.log2(prob)))

def brightness(image_rgb):
    return float(np.mean(to_gray(image_rgb)))

def contrast(image_rgb):
    return float(np.std(to_gray(image_rgb)))

def colorfulness(image_rgb):
    image = image_rgb.astype(np.float32)
    rg = np.abs(image[:, :, 0] - image[:, :, 1])
    yb = np.abs(0.5 * (image[:, :, 0] + image[:, :, 1]) - image[:, :, 2])
    return float(np.sqrt(np.std(rg) ** 2 + np.std(yb) ** 2) + 0.3 * np.sqrt(np.mean(rg) ** 2 + np.mean(yb) ** 2))

def all_metrics(original, enhanced, reference=None, runtime_ms=np.nan):
    data = {
        "brightness": brightness(enhanced),
        "contrast": contrast(enhanced),
        "entropy": entropy(enhanced),
        "colorfulness": colorfulness(enhanced),
        "brightness_gain": brightness(enhanced) - brightness(original),
        "contrast_gain": contrast(enhanced) - contrast(original),
        "runtime_ms": runtime_ms,
    }
    if reference is not None:
        data["mse"] = mse(enhanced, reference)
        data["psnr"] = psnr(enhanced, reference)
        data["ssim"] = ssim_score(enhanced, reference)
    else:
        data["mse"] = np.nan
        data["psnr"] = np.nan
        data["ssim"] = np.nan
    return data


# Part J - Run Experiments


In [ ]:
METHODS = {
    "global_he": global_he,
    "clahe_ycrcb": clahe_ycrcb,
    "gamma_clahe": gamma_clahe,
    "single_scale_retinex": single_scale_retinex,
    "multi_scale_retinex": multi_scale_retinex,
    "lime_paper": lime_paper_method,
    "improved_lime": improved_lime_method,
}

results = {}
rows = []

for image_path in list_images(INPUT_DIR):
    original = read_rgb(image_path)
    ref_path = reference_for(image_path)
    reference = read_rgb(ref_path) if ref_path else None
    results[image_path.name] = {"original": original, "reference": reference, "methods": {}}

    for method_name, method in METHODS.items():
        start = time.perf_counter()
        enhanced = method(original)
        runtime_ms = (time.perf_counter() - start) * 1000
        results[image_path.name]["methods"][method_name] = enhanced
        save_rgb(OUTPUT_DIR / method_name / image_path.name, enhanced)
        row = {"image": image_path.name, "method": method_name}
        row.update(all_metrics(original, enhanced, reference, runtime_ms))
        rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(TABLE_DIR / "all_metrics.csv", index=False)
metrics_df.head()


## Average Method Comparison


In [ ]:
average_df = metrics_df.groupby("method").mean(numeric_only=True).reset_index()
sort_col = "ssim" if average_df["ssim"].notna().any() else "entropy"
average_df = average_df.sort_values(sort_col, ascending=False)
average_df.to_csv(TABLE_DIR / "average_method_comparison.csv", index=False)
average_df


# Part K - Visual Results and Paper Methodology Evidence


In [ ]:
def show_grid(items, title, cols=None, size=(18, 7)):
    if cols is None:
        cols = len(items)
    rows = math.ceil(len(items) / cols)
    plt.figure(figsize=size)
    for i, (label, image) in enumerate(items, start=1):
        plt.subplot(rows, cols, i)
        plt.imshow(image, cmap="gray" if image.ndim == 2 else None)
        plt.title(label)
        plt.axis("off")
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

first_name = list(results.keys())[0]
first_original = results[first_name]["original"]
lime_out, lime_debug = lime_paper_method(first_original, return_debug=True)
improved_out, improved_debug = improved_lime_method(first_original, return_debug=True)

show_grid(
    [
        ("Input", first_original),
        ("Initial T_hat", lime_debug["initial_illumination"]),
        ("Refined T", lime_debug["refined_illumination"]),
        ("LIME Output", lime_out),
        ("Improved LIME", improved_out),
    ],
    "LIME Methodology Intermediate Outputs",
    cols=5,
    size=(20, 5),
)


In [ ]:
comparison_methods = ["original", "global_he", "clahe_ycrcb", "gamma_clahe", "multi_scale_retinex", "lime_paper", "improved_lime"]

for image_name, data in list(results.items())[:5]:
    items = []
    for method_name in comparison_methods:
        if method_name == "original":
            items.append(("original", data["original"]))
        else:
            items.append((method_name, data["methods"][method_name]))
    show_grid(items, f"Visual Comparison - {image_name}", cols=len(items), size=(22, 5))


## Histogram and CDF Comparison


In [ ]:
def gray_hist(image):
    return cv2.calcHist([to_gray(image)], [0], None, [256], [0, 256]).ravel()

def gray_cdf(image):
    hist = gray_hist(image)
    cdf = np.cumsum(hist)
    return cdf / cdf[-1]

for image_name, data in list(results.items())[:3]:
    plt.figure(figsize=(14, 5))
    for method_name in ["original", "lime_paper", "improved_lime", "clahe_ycrcb"]:
        image = data["original"] if method_name == "original" else data["methods"][method_name]
        plt.plot(np.log1p(gray_hist(image)), label=method_name)
    plt.title(f"Log Histogram Comparison - {image_name}")
    plt.xlabel("Intensity")
    plt.ylabel("log(1 + frequency)")
    plt.legend()
    plt.show()

    plt.figure(figsize=(14, 5))
    for method_name in ["original", "lime_paper", "improved_lime", "clahe_ycrcb"]:
        image = data["original"] if method_name == "original" else data["methods"][method_name]
        plt.plot(gray_cdf(image), label=method_name)
    plt.title(f"CDF Comparison - {image_name}")
    plt.xlabel("Intensity")
    plt.ylabel("Cumulative Probability")
    plt.legend()
    plt.show()


## Runtime and Metric Graphs


In [ ]:
plot_metrics = ["brightness", "contrast", "entropy", "brightness_gain", "contrast_gain", "runtime_ms"]
if average_df["ssim"].notna().any():
    plot_metrics.extend(["psnr", "ssim"])

for metric in plot_metrics:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=average_df, x="method", y=metric)
    plt.title(f"Average {metric} by Method")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


# Part L - Discussion, Innovation, and Viva


## What Came From the Research Paper?

- Initial illumination estimation using maximum RGB channel.
- Illumination refinement using structure-aware smoothing idea.
- Enhancement by illumination correction using gamma.
- Visual proof through illumination maps.

## What Is Our Improvement?

- Bilateral denoising before LIME to reduce low-light noise.
- Adaptive gamma selection based on input brightness.
- CLAHE on luminance after LIME to improve local contrast.

## Expected Discussion

LIME should improve dark regions while preserving structures better than simple histogram equalization. Improved LIME may provide stronger contrast and visibility, but may require slightly more runtime.


In [ ]:
viva_questions = [
    ("Why did you select LIME?", "It is an IEEE TIP paper and directly solves low-light enhancement using illumination map estimation."),
    ("What is T_hat?", "It is the initial illumination map estimated as the maximum of R, G, and B channels at each pixel."),
    ("Why refine illumination?", "Raw illumination can contain noise and texture; refinement smooths illumination while preserving structure."),
    ("What is your innovation?", "We add denoising, adaptive gamma, and CLAHE after LIME."),
    ("How did you evaluate results?", "Using visual comparison, histograms, brightness, contrast, entropy, runtime, and PSNR/SSIM when references exist."),
]

for question, answer in viva_questions:
    print("Q:", question)
    print("A:", answer)
    print()


## Student-Wise Contribution


In [ ]:
contributions = {
    "Student 1 - Research and Algorithm": [
        "Selected and explained LIME paper",
        "Implemented LIME methodology",
        "Explained HE, CLAHE, Retinex, and illumination maps",
    ],
    "Student 2 - Dataset and Evaluation": [
        "Handled image upload/dataset organization",
        "Implemented PSNR, SSIM, MSE, entropy, brightness, contrast, runtime",
        "Generated result tables and metric graphs",
    ],
    "Student 3 - Visualization and Documentation": [
        "Created visual comparisons, histograms, CDF plots, and illumination maps",
        "Prepared viva discussion and output ZIP",
        "Maintained report-ready figures and explanations",
    ],
}

for student, tasks in contributions.items():
    print(student)
    for task in tasks:
        print(" -", task)
    print()


## Download Final Results


In [ ]:
zip_path = Path("/content/lime_low_light_results.zip") if IN_COLAB else Path("lime_low_light_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for path in BASE_DIR.rglob("*"):
        if path.is_file():
            zipf.write(path, path.relative_to(BASE_DIR))

print("Created:", zip_path)
if IN_COLAB:
    files.download(str(zip_path))


# References

[1] X. Guo, Y. Li, and H. Ling, "LIME: Low-Light Image Enhancement via Illumination Map Estimation," *IEEE Transactions on Image Processing*, vol. 26, no. 2, pp. 982-993, 2017.  
[2] K. Zuiderveld, "Contrast Limited Adaptive Histogram Equalization," in *Graphics Gems IV*, Academic Press, 1994.  
[3] E. H. Land, "The Retinex Theory of Color Vision," *Scientific American*, 1977.  
[4] OpenCV Documentation, "Histograms - Histogram Equalization and CLAHE."  
[5] W. Wei, W. Wang, W. Yang, and J. Liu, "Deep Retinex Decomposition for Low-Light Enhancement," BMVC, 2018.
